## US-12

In [1]:
from pathlib import Path
import runpy
import pandas as pd
ROOT = Path('/home/jovyan/work')
validation = runpy.run_path(str(ROOT/'pipelines/validation_us12_us16.py'))
us12 = validation['us12']()
display(pd.DataFrame(us12['volumes'].items(), columns=['Table Bronze', 'Lignes']))
display(pd.DataFrame([{'Table': t, 'Contrôle': c['nom'], 'Résultat': c['resultat'], 'Valeur': c['valeurs'], 'Code retour': s['code_retour']} for t,s in us12['scans'].items() for c in s['checks']]))
print('Synthèse :', us12['synthese'], '| Données inchangées :', us12['donnees_inchangees'])


,Table Bronze,Lignes
0,crm_client,100000
1,crm_contrat,100000
2,crm_sinistre,100000
3,crm_type_contrat,12
4,cif_clients,40000
5,cif_compte,45000
6,cif_pays,250


,Table,Contrôle,Résultat,Valeur,Code retour
0,client,RQ-003 telephone du souscripteur renseigne BRONZE,fail,[5989],2
1,client,RQ-005 code postal sur 5 chiffres (France) BRONZE,fail,[1456],2
2,client,RQ-007 pays dans la liste de reference BRONZE,fail,[3050],2
3,client,RQ-015 prospect sans crm_contrat BRONZE,fail,[479],2
4,client,RQ-020 BRONZE duplicate_count(client_id) = 0,pass,[0],2
5,client,RQ-020 BRONZE missing_count(client_id) = 0,pass,[0],2
6,client,RQ-009 type de personne autorise BRONZE,fail,[200],2
7,contrat,RQ-001 date de fin posterieure a la date de de...,fail,[800],2
8,contrat,RQ-011 type de crm_contrat connu BRONZE,fail,[200],2
9,contrat,RQ-012 souscripteur existant BRONZE,fail,[1000],2


Synthèse : {'pass': 8, 'warn': 0, 'fail': 13} | Données inchangées : True


In [2]:
display(pd.DataFrame([{'Scénario': cas, 'Code retour': r['code_retour'], 'Résultat': r['checks'][0]['resultat']} for cas,r in us12['demonstrations'].items()]))
print('Configurations CRM / Bronze / Silver / Gold :', us12['sources'])


,Scénario,Code retour,Résultat
0,pass,0,pass
1,warn,1,warn
2,fail,2,fail


Configurations CRM / Bronze / Silver / Gold : {'crm': {'connexion': 'OK', 'tables': 4}, 'bronze': {'connexion': 'OK', 'tables': 7}, 'silver': {'connexion': 'OK', 'tables': 4}, 'gold': {'connexion': 'OK', 'tables': 2}}


## US-16

In [3]:
us16 = validation['us16']()
colonnes = ['regle','relation','testees','nuls','orphelins','anomalies','taux_echec','soda']
display(pd.DataFrame(us16['relations'])[colonnes])
assert us16['concordance_sql_soda']
print('Concordance SQL / Soda vérifiée pour les sept relations.')


,regle,relation,testees,nuls,orphelins,anomalies,taux_echec,soda
0,RQ-012,crm_contrat.souscripteur_id → bronze.crm_clien...,100000,0,1000,1000,1.00,1000.0
1,RQ-013 brut,crm_contrat.assure_id → bronze.crm_client.clie...,100000,0,40300,40300,40.30,40300.0
2,RQ-013,crm_contrat.assure_id → bronze.crm_client.clie...,100000,0,300,300,0.30,300.0
3,RQ-011,crm_contrat.code_type → bronze.crm_type_contra...,100000,0,200,200,0.20,200.0
4,Lien sinistre,crm_sinistre.contrat_id → bronze.crm_contrat.c...,100000,0,0,0,0.00,0.0
5,Lien CIF,cif_compte.column1 → bronze.cif_clients.column0,45000,0,0,0,0.00,0.0
6,RQ-007,crm_client.pays → silver.ref_pays.code_pays,100000,0,3050,3050,3.05,3050.0


Concordance SQL / Soda vérifiée pour les sept relations.


In [4]:
display(pd.DataFrame([us16['pays']]))
display(pd.DataFrame([us16['reference_pays']]))
print('RQ-007 : comparaison exacte ; NULL en échec, espaces conservés.')
display(pd.DataFrame(us16['fixture_rq007']))
print('Preuves : reports/s3/validation_us12_us16/')


,testees,nuls,vides_ou_blancs,espaces_exterieurs,corrigeables_par_trim
0,100000,0,400,0,0


,n,distincts,nuls
0,250,250,0


RQ-007 : comparaison exacte ; NULL en échec, espaces conservés.


,id,ko
0,1,1
1,2,0
2,3,1
3,4,1
4,5,1
5,6,1
6,7,1
7,8,1


Preuves : reports/s3/validation_us12_us16/
